# PipelineSLA-Telemetry — Pipeline Audit WalkthroughAn end-to-end operational audit of the enterprise ETL estate, rendered withinteractive Plotly charts directly inside the **VS Code Notebook editor**.**What this notebook answers, in order:**1. Is the estate healthy this week? (reliability scorecard)2. Which tasks are drifting? (Weibull percentile boundaries)3. Will tonight's run breach the 07:00 SLA? (probabilistic forecast)4. Did any pipeline silently load zero rows? (two-sided CUSUM)5. If a task stalls, whose dashboards go dark? (blast radius)

In [ ]:
# %% Cell 1 — Bootstrap the warehousefrom __future__ import annotationsimport sysfrom pathlib import Pathimport pandas as pdROOT = Path.cwd()if (ROOT / "src").exists():    sys.path.insert(0, str(ROOT / "src"))from pipeline_sla.config import load_settingsfrom pipeline_sla.storage.warehouse import TelemetryWarehousesettings = load_settings()db_path = ROOT / settings.storage.database_pathprint("warehouse:", db_path, "| exists:", db_path.exists())warehouse = TelemetryWarehouse(db_path, read_only=db_path.exists())warehouse.counts()

In [ ]:
# %% Cell 2 — Weekly reliability scorecardfrom pipeline_sla.analytics.metrics import operational_summaryoverview = operational_summary(warehouse)overview

In [ ]:
# %% Cell 3 — Weibull + Log-Normal duration fitsfrom pipeline_sla.analytics.weibull_forecaster import WeibullForecasterDAG, TASK = "gl_financial_reconciliation", "reconcile_subledgers"durations = warehouse.get_task_durations(DAG, TASK)forecaster = WeibullForecaster().fit(durations)pd.DataFrame([f.to_dict() for f in forecaster.fits.values()])

In [ ]:
# %% Cell 4 — Interactive breach-probability curvefrom pipeline_sla.dashboard import viewsfig = views.weibull_probability_figure(    durations, elapsed_seconds=900, cutoff_seconds=1320,    title=f"Weibull SLA Breach Forecast — {TASK}",)fig.show()

In [ ]:
# %% Cell 5 — Two-sided CUSUM: silent zero-row ingestionfrom pipeline_sla.analytics.cusum_detector import VolumeAnomalyDetectordetector = VolumeAnomalyDetector()series = warehouse.get_volume_series("sales_ingestion_etl", "load_sales_fact")detected = detector.detect(series)detected[detected["is_silent_empty"]][["execution_date", "rows_loaded", "severity", "reason"]]

In [ ]:
# %% Cell 6 — Volume vs expected baseline with alarm markersviews.volume_anomaly_figure(series).show()

In [ ]:
# %% Cell 7 — CUSUM drift monitorviews.cusum_figure(series).show()

In [ ]:
# %% Cell 8 — Downstream blast radiusfrom pipeline_sla.lineage.dag_graph import LineageGraphfrom pipeline_sla.lineage.blast_radius import BlastRadiusCalculatorgraph = LineageGraph.from_settings(settings)impact = BlastRadiusCalculator(graph).compute("reconcile_subledgers")print(f"Blast radius {impact.score:,.0f} — {impact.category}")pd.DataFrame([d.to_dict() for d in impact.impacted])

In [ ]:
# %% Cell 9 — Interactive lineage mapviews.lineage_figure(graph, highlight_node="reconcile_subledgers").show()